# Prepare MATH-500 for prompt optimization

This notebook loads the canonical HuggingFaceH4 MATH-500 test file, shows five examples, converts all 500 problems to the project's shared schema, validates uniqueness and required fields, and writes the processed files under `data/processed/math500/`. It then runs `prepare_math500_training.py` to create a fixed 6,599/900 train/validation split and three subject-and-level-stratified validation folds of 300.

MATH-500 itself is kept strictly as a test set. Training and validation come only from the original MATH training split. Answers may be symbolic expressions, fractions, tuples, intervals, or sets, so they must not be reduced to digits only during evaluation.

In [1]:
import hashlib
import json
import re
from collections import Counter
from pathlib import Path

In [2]:
def find_repo_root():
    """Find the repository root from the current notebook directory."""
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "AGENTS.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


def read_jsonl(path):
    """Read non-empty JSON objects from a JSONL file."""
    with path.open(encoding="utf-8") as stream:
        return [json.loads(line) for line in stream if line.strip()]


def show_examples(records, count=5):
    """Display a small number of records in readable JSON."""
    print(json.dumps(records[:count], ensure_ascii=False, indent=2))


def file_sha256(path):
    """Calculate the SHA-256 checksum of a local file."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_question(text):
    """Normalize whitespace and letter case for duplicate checks."""
    return re.sub(r"\s+", " ", text).strip().casefold()

In [3]:
REPO_ROOT = find_repo_root()
RAW_PATH = REPO_ROOT / "data/math500/original/test.jsonl"
OUTPUT_DIR = REPO_ROOT / "data/processed/math500"
SOURCE_DATASET = "HuggingFaceH4/MATH-500"
SOURCE_REVISION = "6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be"
SOURCE_URL = f"https://huggingface.co/datasets/{SOURCE_DATASET}/resolve/{SOURCE_REVISION}/test.jsonl"
EXPECTED_SHA256 = "35dc41080a3680858b27fa7e0533d2d547825316fc5dafe5d316f4ccc5a06132"

raw_records = read_jsonl(RAW_PATH)
raw_sha256 = file_sha256(RAW_PATH)
assert raw_sha256 == EXPECTED_SHA256
print(f"Raw test records: {len(raw_records):,}")
print(f"SHA-256: {raw_sha256}")

Raw test records: 500
SHA-256: 35dc41080a3680858b27fa7e0533d2d547825316fc5dafe5d316f4ccc5a06132


## Five original examples

In [4]:
show_examples(raw_records, count=5)

[
  {
    "problem": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$",
    "solution": "We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n\n[asy]\nunitsize(0.8 cm);\n\ndraw((-0.5,0)--(3.5,0));\ndraw((0,-0.5)--(0,3.5));\ndraw(arc((0,0),3,0,90),red,Arrow(6));\n\ndot((0,3), red);\nlabel(\"$(0,3)$\", (0,3), W);\ndot((3,0), red);\n[/asy]\n\nTherefore, the polar coordinates are $\\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}.$",
    "answer": "\\left( 3, \\frac{\\pi}{2} \\right)",
    "subject": "Precalculus",
    "level": 2,
    "unique_id": "test/precalculus/807.json"
  },
  {
    "problem": "Define\n\\[p = \\sum_{k = 1}^\\infty \\frac{1}{k^2} \\quad \\text{and} \\quad q = \\sum_{k = 1}^\\infty \\frac{1}{k^3}.\\]Find a way to write\n\\[\\sum_{j = 1

In [5]:
def make_math500_id(source_id):
    """Create a stable project ID from the original problem path."""
    source_slug = source_id.removesuffix(".json").replace("/", "-")
    return f"math500-{source_slug}"


def normalize_math500(row):
    """Convert one MATH-500 row to the prompt-optimization schema."""
    return {
        "id": make_math500_id(row["unique_id"]),
        "dataset": "math500",
        "task_type": "math_symbolic_answer",
        "split": "test",
        "question": row["problem"].strip(),
        "answer": row["answer"].strip(),
        "solution": row["solution"].strip(),
        "subject": row["subject"].strip(),
        "level": int(row["level"]),
        "source_id": row["unique_id"],
    }


def validate_math500(records):
    """Check size, uniqueness, fields, levels, and split labels."""
    assert len(records) == 500
    assert len({record["id"] for record in records}) == len(records)
    question_keys = [normalize_question(record["question"]) for record in records]
    assert len(set(question_keys)) == len(question_keys)
    assert all(record["split"] == "test" for record in records)
    assert all(record["question"] and record["answer"] and record["solution"] for record in records)
    assert all(record["subject"] and 1 <= record["level"] <= 5 for record in records)
    subject_counts = dict(sorted(Counter(record["subject"] for record in records).items()))
    level_counts = dict(sorted(Counter(record["level"] for record in records).items()))
    return subject_counts, level_counts


prepared_records = [normalize_math500(row) for row in raw_records]
subject_counts, level_counts = validate_math500(prepared_records)
print("Validation passed.")
print("Subjects:", subject_counts)
print("Levels:", level_counts)

Validation passed.
Subjects: {'Algebra': 124, 'Counting & Probability': 38, 'Geometry': 41, 'Intermediate Algebra': 97, 'Number Theory': 62, 'Prealgebra': 82, 'Precalculus': 56}
Levels: {1: 43, 2: 90, 3: 105, 4: 128, 5: 134}


## Five prepared examples

In [6]:
show_examples(prepared_records, count=5)

[
  {
    "id": "math500-test-precalculus-807",
    "dataset": "math500",
    "task_type": "math_symbolic_answer",
    "split": "test",
    "question": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$",
    "answer": "\\left( 3, \\frac{\\pi}{2} \\right)",
    "solution": "We have that $r = \\sqrt{0^2 + 3^2} = 3.$  Also, if we draw the line connecting the origin and $(0,3),$ this line makes an angle of $\\frac{\\pi}{2}$ with the positive $x$-axis.\n\n[asy]\nunitsize(0.8 cm);\n\ndraw((-0.5,0)--(3.5,0));\ndraw((0,-0.5)--(0,3.5));\ndraw(arc((0,0),3,0,90),red,Arrow(6));\n\ndot((0,3), red);\nlabel(\"$(0,3)$\", (0,3), W);\ndot((3,0), red);\n[/asy]\n\nTherefore, the polar coordinates are $\\boxed{\\left( 3, \\frac{\\pi}{2} \\right)}.$",
    "subject": "Precalculus",
    "level": 2,
    "source_id": "test/precalculus/807.json"
  },
  {
    "id": "math500-test-intermediate_algebr

In [7]:
def write_jsonl(path, records):
    """Write records as one JSON object per line."""
    with path.open("w", encoding="utf-8") as stream:
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_json(path, value):
    """Write a JSON value with readable indentation."""
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_jsonl(OUTPUT_DIR / "test.jsonl", prepared_records)
write_json(OUTPUT_DIR / "all.json", prepared_records)
write_json(
    OUTPUT_DIR / "dataset_info.json",
    {
        "dataset": "math500",
        "task_type": "math_symbolic_answer",
        "splits": {"test": len(prepared_records)},
        "subject_counts": subject_counts,
        "level_counts": {str(key): value for key, value in level_counts.items()},
        "source": {
            "dataset": SOURCE_DATASET,
            "revision": SOURCE_REVISION,
            "url": SOURCE_URL,
            "raw_file": str(RAW_PATH.relative_to(REPO_ROOT)),
            "sha256": raw_sha256,
        },
        "files": ["test.jsonl", "all.json"],
    },
)
print(f"Saved {len(prepared_records):,} test records to {OUTPUT_DIR}")

# Prepare MATH training and validation after the canonical test conversion.
import runpy
runpy.run_path(str(REPO_ROOT / "codes/prepare_math500_training.py"), run_name="__main__")

Saved 500 test records to /storage2/home/aunabilchakma/codes/RE_Prompt_optimizer/data/processed/math500
